# **Machine Learning Models & Evaluation**

This section focuses on training and evaluating multiple machine learning models to predict loan default risk.

We start with a simple baseline model and progressively move toward more advanced ensemble methods.

The goal is to:
- Compare model performance
- Handle class imbalance
- Improve predictive power
- Select the best final model

Models evaluated:
- Logistic Regression (Baseline)
- Random Forest (Ensemble Model)
- XGBoost (Advanced Gradient Boosting)

Evaluation metrics include:
- Confusion Matrix
- Precision, Recall, F1-score
- ROC-AUC Score

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
import joblib



**Load Clean Data**

In [2]:
df = pd.read_csv("../data/credit_risk_clean.csv")
df.head()

,person_age,person_income,person_emp_length,loan_grade,loan_amnt,loan_int_rate,loan_status,loan_percent_income,cb_person_default_on_file,cb_person_cred_hist_length,person_home_ownership_OTHER,person_home_ownership_OWN,person_home_ownership_RENT,loan_intent_EDUCATION,loan_intent_HOMEIMPROVEMENT,loan_intent_MEDICAL,loan_intent_PERSONAL,loan_intent_VENTURE
0,22,59000,17.0,3,35000,16.02,1,0.59,1,3,0,0,1,0,0,0,1,0
1,21,9600,5.0,1,1000,11.14,0,0.10,0,2,0,1,0,1,0,0,0,0
2,25,9600,1.0,2,5500,12.87,1,0.57,0,3,0,0,0,0,0,1,0,0
3,23,65500,4.0,2,35000,15.23,1,0.53,0,2,0,0,1,0,0,1,0,0
4,24,54400,8.0,2,35000,14.27,1,0.55,1,4,0,0,1,0,0,1,0,0


**Defining Features & Target**

In [3]:
X = df.drop('loan_status', axis=1)
y = df['loan_status']

**Train / Test Split**

In [4]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

**Standardization**

In [5]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

**Baseline Model: Logistic Regression**

The target variable shows a moderate class imbalance (approximately 78% of loans are repaid versus 22% defaults).

In this context, Logistic Regression can be biased toward the majority class. To address this imbalance without altering the original data distribution, we use the parameter class_weight='balanced'.

This approach assigns a higher weight to the minority class (defaults), improving the model’s ability to detect risky borrowers.

We do not apply techniques such as SMOTE because the imbalance is not extreme, and tree-based models used later (such as Random Forest and XGBoost) are naturally more robust to class imbalance.

In [6]:
lr = LogisticRegression(
    class_weight='balanced',
    max_iter=1000,
    random_state=42
)

lr.fit(X_train_scaled, y_train)

y_pred_lr = lr.predict(X_test_scaled)
y_proba_lr = lr.predict_proba(X_test_scaled)[:, 1]



In [7]:
print("LOGISTIC REGRESSION RESULTS")

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_lr))

print("\nKey Metrics (Class 1 - Default):")
print(classification_report(y_test, y_pred_lr))

print("\nROC-AUC:", roc_auc_score(y_test, y_proba_lr))


LOGISTIC REGRESSION RESULTS

Confusion Matrix:
[[4051 1044]
 [ 309 1113]]

Key Metrics (Class 1 - Default):
              precision    recall  f1-score   support

           0       0.93      0.80      0.86      5095
           1       0.52      0.78      0.62      1422

    accuracy                           0.79      6517
   macro avg       0.72      0.79      0.74      6517
weighted avg       0.84      0.79      0.81      6517


ROC-AUC: 0.8667683907308259


The Logistic Regression model shows strong performance in identifying default cases, with a recall of 0.78, meaning it successfully detects most risky borrowers. However, the precision of 0.52 indicates a moderate number of false positives, meaning some safe borrowers are incorrectly classified as risky. This trade-off is expected due to the use of class weighting to handle class imbalance. Overall, the model achieves a good balance between sensitivity and precision, with a strong ROC-AUC score of 0.86, indicating good overall discriminative power. As a result, Logistic Regression provides a solid baseline for the credit risk prediction task.

**Random Forest Model**

In [8]:
rf = RandomForestClassifier(
    n_estimators=200,
    max_depth=None,
    min_samples_split=2,
    min_samples_leaf=1,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1
)

rf.fit(X_train, y_train)

RandomForestClassifier(class_weight='balanced', n_estimators=200, n_jobs=-1,
                       random_state=42)

In [9]:
y_pred_rf = rf.predict(X_test)
y_proba_rf = rf.predict_proba(X_test)[:, 1]

In [10]:
print("RANDOM FOREST RESULTS")

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_rf))

print("\nClassification Report:")
print(classification_report(y_test, y_pred_rf))

print("\nROC-AUC Score:")
print(roc_auc_score(y_test, y_proba_rf))

RANDOM FOREST RESULTS

Confusion Matrix:
[[5075   20]
 [ 407 1015]]

Classification Report:
              precision    recall  f1-score   support

           0       0.93      1.00      0.96      5095
           1       0.98      0.71      0.83      1422

    accuracy                           0.93      6517
   macro avg       0.95      0.85      0.89      6517
weighted avg       0.94      0.93      0.93      6517


ROC-AUC Score:
0.9340338077235756


The Random Forest model shows strong overall performance with an ROC-AUC of 0.93, indicating excellent ability to distinguish between default and non-default cases. Compared to Logistic Regression, the model achieves significantly higher precision (0.98), meaning that when it predicts a default, it is almost always correct. However, this comes at the cost of lower recall (0.71), meaning that some actual default cases are not detected. This reflects a more conservative model behavior, favoring precision over sensitivity. Overall, Random Forest provides a strong and stable model for credit risk prediction.

Random Forest improves overall performance compared to Logistic Regression, especially in reducing false positives, but sacrifices some ability to detect all risky borrowers.

**XGBoost Model (State-of-the-Art for Tabular Data)**

XGBoost (Extreme Gradient Boosting) is an advanced ensemble method that builds decision trees sequentially, where each new tree corrects the errors of the previous ones.

In [11]:
param_dist = {
    'n_estimators': [100, 200, 300],
    'max_depth': [3, 5, 7],
    'learning_rate': [0.01, 0.05, 0.1],
    'subsample': [0.7, 0.8, 1.0],
    'colsample_bytree': [0.7, 0.8, 1.0]
}

In [12]:
scale_pos_weight = len(y_train[y_train == 0]) / len(y_train[y_train == 1])
scale_pos_weight

3.5838902567710167

“missing a default is 3.5x worse than missing a safe loan”

In [13]:
xgb = XGBClassifier(
    scale_pos_weight=scale_pos_weight,
    random_state=42,
    eval_metric='logloss'
)
random_search = RandomizedSearchCV(
    estimator=xgb,
    param_distributions=param_dist,
    n_iter=20,
    scoring='roc_auc',
    cv=3,
    verbose=1,
    random_state=42,
    n_jobs=-1
)

random_search.fit(X_train, y_train)
print(random_search.best_params_)


Fitting 3 folds for each of 20 candidates, totalling 60 fits
{'subsample': 1.0, 'n_estimators': 300, 'max_depth': 5, 'learning_rate': 0.1, 'colsample_bytree': 0.8}


In [14]:
best_xgb = random_search.best_estimator_

y_pred_best = best_xgb.predict(X_test)
y_proba_best = best_xgb.predict_proba(X_test)[:,1]

print("XGBoost RESULTS")

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_best))

print("\nClassification Report:")
print(classification_report(y_test, y_pred_best))

print("\nROC-AUC Score:")

print("ROC-AUC:", roc_auc_score(y_test, y_proba_best))

XGBoost RESULTS

Confusion Matrix:
[[4857  238]
 [ 277 1145]]

Classification Report:
              precision    recall  f1-score   support

           0       0.95      0.95      0.95      5095
           1       0.83      0.81      0.82      1422

    accuracy                           0.92      6517
   macro avg       0.89      0.88      0.88      6517
weighted avg       0.92      0.92      0.92      6517


ROC-AUC Score:
ROC-AUC: 0.9528984457059884


The XGBoost model achieves the best overall performance among all tested models, with a ROC-AUC score of 0.95, indicating excellent discriminative ability. It provides a strong balance between precision (0.83) and recall (0.81), meaning it effectively detects most default cases while maintaining a reasonable number of false positives. Compared to Logistic Regression and Random Forest, the tuned XGBoost model slightly improves both recall and overall stability, reducing missed defaults while keeping false alarms under control. This makes it the most suitable model for the credit risk prediction task.

In [15]:
joblib.dump(best_xgb, "../models/xgboost_model.pkl")

['../models/xgboost_model.pkl']